In [1]:
import os
import pandas as pd
import warnings
import numpy as np
import matplotlib.pyplot as plt
remove_attention_failers = 0

In [2]:
test = []
def extract_basic_info(csv_path):
    df = pd.read_csv(csv_path)
    

    free_texts = df['explanation_box.text'].dropna().tolist()


    # Feedback (free text response about experiment)
    if 'feedback_text.text' in df.columns:
        feedback = df['feedback_text.text'].dropna().tolist()
    else:
        print(f"Missing 'feedback_text.text' in {os.path.basename(csv_path)}")
        fail_text = f"Missing 'feedback_text.text' in {os.path.basename(csv_path)}"
        test.append(fail_text)
        feedback = []

    #Actual fertility score (fertility on each training trial)
    fertility_score = df['fertility_score'].dropna().tolist() [:-1]
    
    # Training trial stop time (time it took to finish the training loop)
    isi_values = df['ISI.stopped'].dropna().tolist()

    #Get the ISI value for the last training trial, store it
    trial_stop_time = isi_values[-1] if isi_values else np.nan

    #First row with a non-empty value in 'images_list', which shows the order of testing images presented
    images_row = df[df['images_list'].notna()].iloc[0] if not df[df['images_list'].notna()].empty else None

    #Turn the images from PNGs to names
    images = [img.split('/')[-1].replace('.png','') for img in images_row['images_list'].split(',')]

    #First row with a non-empty value in 'sliderRatings', which shows the ratings for testing images
    ratings_row = df[df['sliderRatings'].notna()].iloc[0] if not df[df['sliderRatings'].notna()].empty else None

    #Turn the ratings into floats split by commas
    ratings = [float(r) for r in ratings_row['sliderRatings'].strip('[]').split(',')]

    #Across the training trials, add information about features (what actual feature values were shown)
    train_feet = df['feet'].dropna().tolist() [:-1]
    train_stripes = df['stripes'].dropna().tolist() [:-1]

    #Across the training trials, add information about feature relevance
    train_categories =  df['category'].dropna().tolist() [:-1]



    #Extracting the relevant and irrelevant feature dimension info
    dims = {}
    cols = ['relevant_dim', 'irrelevant_dim', 
            'feet_high', 'stripes_low', 
            'stripes_high', 'feet_low']

    for col in cols:
        vals = df[col].dropna().unique()

        if len(vals) == 0:
            dims[col] = np.nan
            print(f"Warning: No values found in {col}")

        elif len(vals) == 1:
            dims[col] = vals[0]

        else:
            warnings.warn(
                f"Multiple values found in {col}: {vals}"
            )
            dims[col] = vals[0] 

    print(dims)

    #Categories for the testing images, in the order shown
    test_categories = images_row['testing_categories'].split(',')

    #Write condition (this is the unique identifier for a certain order of training trials)
    condition = images_row['condition'] if images_row is not None and 'condition' in images_row else np.nan

    #Add in the order of images during training
    training_image_order = [img.split('/')[-1].replace('.png','') 
                        for img in df['image_file'].dropna().tolist()] [:-1]
    
    #Updated code to get slider responses (subjective reports of feature relevance)
    df['feature_clean'] = (
    df['feature']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'stripe': 'stripes'})
)
    slider_responses = {}
    features = ['feet', 'stripes']

    for feat in features:
        sub = df[df['feature_clean'] == feat]

        # 1. discrete
        disc = sub['discrete_slider.response']
        disc_val = disc[disc.notna() & (disc != "")].iloc[0] if (disc.notna() & (disc != "")).any() else np.nan
        print(disc_val)
        slider_responses[f'{feat}_discrete_slider.response'] = disc_val

        # 2. direction
        col = 'direction_response_label'

        if col not in sub.columns:
            warnings.warn(
                f"Column '{col}' not found for feature '{feat}' in {os.path.basename(csv_path)}"
            )
            dirc_val = np.nan
        else:
            dirc = sub[col]
            dirc_val = (
                dirc[dirc.notna() & (dirc != "")].iloc[0]
                if (dirc.notna() & (dirc != "")).any() and disc_val != 'No'
                else np.nan
            )

        slider_responses[f'{feat}_direction_slider.response'] = dirc_val


        # 3. continuous
        cont = sub['continuous_slider.response']
        cont_val = cont[cont.notna() & (cont != "")].iloc[0] if (cont.notna() & (cont != "")).any() and disc_val != 'No' else np.nan #only store continuous if discrete was "Yes"
        print(cont_val)
        slider_responses[f'{feat}_continuous_slider.response'] = cont_val

    #Adding attention check result
    att_rows = df[df['button_3_correct.numClicks'].notna()]
    if not att_rows.empty:
        att_val = att_rows.iloc[0]['button_3_correct.numClicks']
    else:
        att_val = np.nan
    attention_check = 1 if att_val == 1 else 0


    #Spontaneous explain/predict ratings:
    cogpro_predict = df['cogpro_predict.response'].dropna().iloc[0]
    cogpro_explain = df['cogpro_explain.response'].dropna().iloc[0]
    cogpro_predict = float(pd.Series([cogpro_predict]).astype(str).str.extract(r'(\d+\.?\d*)').iloc[0,0])
    cogpro_explain = float(pd.Series([cogpro_explain]).astype(str).str.extract(r'(\d+\.?\d*)').iloc[0,0])


    result = {
        'participant': os.path.basename(csv_path)[:3],
        'free_texts': free_texts,
        'feedback': feedback,
        'fertility_score': fertility_score,
        'trial_stop_time': trial_stop_time,
        'testing_image_order': images,
        'testing_responses': ratings,
        'training_categories': train_categories,
        'training_feet': train_feet,
        'training_stripes': train_stripes,
        'testing_categories': test_categories,
        'conditionOrder': condition,
        'training_image_order': training_image_order,
        'attention_check': attention_check,
        'cogpro_predict': cogpro_predict,
        'cogpro_explain': cogpro_explain,
        **dims
        
    }
    result.update(slider_responses)
    return result

topdir = '/Users/sm6511/Desktop/Prediction-Accomodation-Exp'
study = 'Study4.0'
dates = [
    '2026-05-18',
    '2026-05-19'
]
datadir = os.path.join(topdir, f'data/{study}/Accommodate')
cleaneddir = os.path.join(topdir, f'data/{study}/Cleaned')
all_participants = []

for fname in os.listdir(datadir):
    if fname.endswith('.csv') and fname:
        participant_id = fname[:3]
        if not any(d in fname for d in dates):
            continue
        csv_path = os.path.join(datadir, fname)
        print(csv_path)
        info = extract_basic_info(csv_path)
        all_participants.append(info)

df_all = pd.DataFrame(all_participants)
if remove_attention_failers:
    df_all = df_all[df_all['attention_check'] == 1]
    df_all.to_csv(os.path.join(cleaneddir, f'{study}AccommodateAttRemoved.csv'), index=False)
else:
    df_all.to_csv(os.path.join(cleaneddir, f'{study}Accommodate.csv'), index=False)

print(df_all[df_all['attention_check'] == 1])
print(test)

/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/199_explain2_2026-05-18_13h33.07.159.csv
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
No
nan
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/208_explain2_2026-05-18_14h12.03.983.csv
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'C'}
Yes
5.0
Yes
6.0
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/079_explain2_2026-05-18_14h15.17.688.csv
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'F', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'C'}
Yes
2.0
Yes
6.0
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/193_explain2_2026-05-18_18h22.50.917.csv
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'C', 'stripes_low': 'E', 'str

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 199_explain2_2026-05-18_13h33.07.159.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 199_explain2_2026-05-18_13h33.07.159.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 146_explain2_2026-05-18_13h52.48.333.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 146_explain2_2026-05-18_13h52.48.333.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direc

{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
No
nan
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/043_explain2_2026-05-18_11h01.11.366.csv
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'F', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'C'}
Yes
7.0
Yes
5.0
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/194_explain2_2026-05-18_13h55.05.805.csv
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
No
nan
Yes
6.0
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/004_explain2_2026-05-18_10h50.03.987.csv
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'C'}
Yes
4.0
Yes
4.0
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Acco

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 138_explain2_2026-05-18_11h37.04.591.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 138_explain2_2026-05-18_11h37.04.591.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 160_explain2_2026-05-18_11h48.08.228.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 160_explain2_2026-05-18_11h48.08.228.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direc

{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'C', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'F'}
Yes
2.0
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/155_explain2_2026-05-18_14h45.14.166.csv
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
No
nan
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/201_explain2_2026-05-18_18h26.14.797.csv
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'C', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'F'}
Yes
5.0
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/086_explain2_2026-05-18_12h49.52.137.csv
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'C', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'F'}
Yes
5.0
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accomm

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 100_explain2_2026-05-18_14h14.00.650.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 100_explain2_2026-05-18_14h14.00.650.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 144_explain2_2026-05-18_13h53.11.124.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 144_explain2_2026-05-18_13h53.11.124.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direc

{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'C', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'F'}
No
nan
Yes
5.0
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/218_explain2_2026-05-18_11h02.27.690.csv
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'C', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'F'}
Yes
5.0
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/084_explain2_2026-05-18_16h38.21.574.csv
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'C', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'F'}
No
nan
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accommodate/169_explain2_2026-05-18_14h48.25.553.csv
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'C', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'F'}
Yes
7.0
No
nan
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study4.0/Accomm

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 063_explain2_2026-05-18_12h47.44.622.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 063_explain2_2026-05-18_12h47.44.622.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 089_explain2_2026-05-18_13h51.55.087.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 089_explain2_2026-05-18_13h51.55.087.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_77127/4133675787.py:106: UserWarning: Column 'direc